**Stage 1: Row-Level Semantic Labelling**

Each DS1 row is independently labelled across the 6 semantic heads from the project overview. `delay = -1` is treated as `Unknown`.

In [1]:
import json
import numpy as np
import pandas as pd

"""
Following KPIs of DS1 dataset
---------------------------------
phy_mcs: Modulation & Coding Scheme
mac_dl_cqi: Channel Quality Indicator
mac_dl_ri: Rank Indicator
mac_dl_pmi: Precoding Matrix Indicator
mac_ul_buffer: UL Buffer Occupancy
mac_n_prb: Physics Resource Block
rsrq: Reference Signal Recieved Quality
rsrp: Reference Signal Recieved Power
rssi: Recieved Signal Strength Indicator
dl_sinr: Signal to Inference Noise Ratio
se: Spectral Efficiency
dl_bler: Downlink Block Error Rate
delay: Latency
"""

RAW_FEATURES = [
    "time", "phy_mcs", "mac_dl_cqi", "mac_dl_ri", "mac_dl_pmi",
    "mac_ul_buffer", "mac_n_prb", "rsrq", "rsrp", "rssi",
    "dl_sinr", "se", "dl_bler", "delay"
]

"""
For Low-Level Semantic Labelling, we've to consider 6 label heads,
link_quality = sinr + rsrp
congestion = mac_n_prb
latency = delay
interference = rerq
reliability = dl_bler
scheduler = mac_dl_cqi
"""

LABEL_HEADS = [
    "link_quality", "congestion", "latency",
    "interference", "reliability", "scheduler"
]

df = pd.read_csv("DS1.csv")


# Conditionals for Row-Level Semantic Labelling
def classify_link_quality(row):
    if row["dl_sinr"] >= 20 and row["rsrp"] >= -89:
        return "Excellent"
    if row["dl_sinr"] >= 18 and row["rsrp"] >= -91:
        return "Good"
    if row["dl_sinr"] >= 15 and row["rsrp"] >= -95:
        return "Fair"
    return "Poor"

def classify_congestion(row):
    if row["mac_n_prb"] >= 80:
        return "Congested"
    if row["mac_n_prb"] >= 60:
        return "Busy"
    if row["mac_n_prb"] >= 30:
        return "Moderate"
    return "Underutilized"

def classify_latency(row):
    if pd.isna(row["delay"]) or row["delay"] < 0:
        return "Unknown"
    if row["delay"] >= 30:
        return "Critical"
    if row["delay"] >= 20:
        return "Risk"
    if row["delay"] >= 10:
        return "Normal"
    return "Realtime"

def classify_interference(row):
    if row["rsrq"] >= -8:
        return "Low"
    if row["rsrq"] >= -10:
        return "Moderate"
    return "Risk"

def classify_reliability(row):
    if row["dl_bler"] >= 5:
        return "Critical"
    if row["dl_bler"] >= 1:
        return "Warning"
    return "Reliable"

def classify_scheduler(row):
    if row["mac_dl_cqi"] >= 9:
        return "Excellent"
    if row["mac_dl_cqi"] >= 7:
        return "Moderate"
    return "Poor"

CLASSIFIERS = {
    "link_quality": classify_link_quality,
    "congestion": classify_congestion,
    "latency": classify_latency,
    "interference": classify_interference,
    "reliability": classify_reliability,
    "scheduler": classify_scheduler,
}

for head, classifier in CLASSIFIERS.items():
    df[head] = df.apply(classifier, axis=1)

semantic_labels = df[["time", *LABEL_HEADS]]
raw_semantic = df[RAW_FEATURES + LABEL_HEADS]

semantic_labels.to_csv("semantic_labels.csv", index=False)
raw_semantic.to_csv("raw_semantic.csv", index=False)

print("semantic_labels:", semantic_labels.shape)
print("raw_semantic:", raw_semantic.shape)
display(semantic_labels.head())


semantic_labels: (5999, 7)
raw_semantic: (5999, 20)


,time,link_quality,congestion,latency,interference,reliability,scheduler
0,1.743169e+12,Excellent,Underutilized,Unknown,Low,Reliable,Excellent
1,1.743169e+12,Excellent,Underutilized,Unknown,Low,Reliable,Excellent
2,1.743169e+12,Excellent,Underutilized,Unknown,Low,Reliable,Excellent
3,1.743169e+12,Excellent,Underutilized,Unknown,Low,Reliable,Excellent
4,1.743169e+12,Excellent,Underutilized,Unknown,Low,Reliable,Excellent
